# Configuration


In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(".env.pacing", override=True)  # Change to ".env.simulation" for simulation data

# Load configuration from .env
simulation_root = os.getenv("SIMULATION_DIR")
converted_dir = os.getenv("CONVERT_TARGET_DIR")
unfiltered_base_dir = os.getenv("UNFILTERED_BASE_DIR")
comparison_checkpoint_dir = os.getenv("COMPARISON_CHECKPOINT_DIR")
comparison_output_dir = os.getenv("COMPARISON_OUTPUT_DIR")
sim_targets_dir = os.getenv("SIM_TARGETS_DIR")
sim_constraints_dir = os.getenv("SIM_CONSTRAINTS_DIR")
filtered_pacing_segments_dir = os.getenv("FILTERED_PACING_SEG_DIR")
output_dir = os.getenv("OUTPUT_DIR")

# Display loaded configuration
print("Configuration loaded from .env:")
print(f"  SIMULATION_DIR: {simulation_root}")
print(f"  CONVERT_TARGET_DIR: {converted_dir}")
print(f"  UNFILTERED_BASE_DIR: {unfiltered_base_dir}")
print(f"  COMPARISON_CHECKPOINT_DIR: {comparison_checkpoint_dir}")
print(f"  COMPARISON_OUTPUT_DIR: {comparison_output_dir}")
print(f"  SIM_TARGETS_DIR: {sim_targets_dir}")
print(f"  SIM_CONSTRAINTS_DIR: {sim_constraints_dir}")
print(f"  FILTERED_PACING_SEG_DIR: {filtered_pacing_segments_dir}")
print(f"  OUTPUT_DIR: {output_dir}")

# Create directories if they don't exist
os.makedirs(output_dir, exist_ok=True)
os.makedirs(converted_dir, exist_ok=True)
os.makedirs(comparison_checkpoint_dir, exist_ok=True)
os.makedirs(comparison_output_dir, exist_ok=True)

from compile.generator import batch_process_pacing, load_bot_pacing_factors
import polars as pl

bot_pairs = [
    ("MCTS", "Bot_MCTS"),
    ("NN", "Bot_NN"),
    # ("Bot_MLP", "Bot_ML_Classification"),
]

pacing_config_filter = {
    "Timer": [30],
    "ActInterval": [0.1],
    "Round": ["BestOf3"],
    "SkillLeft": ["Boost"],
    "SkillRight": ["Boost"],
}

pacing_factor_bin_size = 1

# Compile Data

## Convert Simulation Log to Parquet / CSV

In [ ]:
from compile.log_to_parquet import ( 
    convert_all_configs
)

convert_all_configs(simulation_root, converted_dir)

## Batch Pacing Segment

In [ ]:
from compile.generator import batch_process_pacing_segments
import os
import time

# filtered_pacing_segments_dir (FILTERED_PACING_SEG_DIR in .pacing.env) is
# checkpoint_dir/name/pacing_segments - derive both back out so this writes to
# the same place load_filtered_target_tracking below reads from.
pacing_segments_checkpoint_dir = os.path.dirname(os.path.dirname(filtered_pacing_segments_dir))
pacing_segments_name = os.path.basename(os.path.dirname(filtered_pacing_segments_dir))

start = time.time()

batch_process_pacing_segments(
    converted_dir,
    batch_size=4,
    checkpoint_dir=pacing_segments_checkpoint_dir,
    name=pacing_segments_name,
    applied_bots=[b for b, _ in bot_pairs],
    config_filter=pacing_config_filter,
    min_pacing=0,
    max_pacing=1,
)

elapsed_seconds = time.time() - start
hours, remainder = divmod(elapsed_seconds, 3600)
minutes, seconds = divmod(remainder, 60)
processing_time = f"{int(hours):02d}:{int(minutes):02d}:{seconds:.2f}"
print(f"\nProcessing Time: {processing_time}")


# Filtered vs Unfiltered Pacing Comparison

Compares MCTS/NN's raw pacing factors (`CollisionRatio`, `AbilityRatio`, `Angle`, `SafeDistance`, `ActionIntensity`, `ActionDensity`, `BotsDistance`, `Velocity` — recomputed offline from `Action`/`Collision` event rows via `compile.generator.batch_process_pacing`, the same formula on both sides) between:

- **Filtered**: MCTS/NN with the dynamic pacing filter applied (`.analytic-cache/converted/pacing_fix3`)
- **Unfiltered**: MCTS/NN's natural behavior with no filter running (`local_setup/simulated-13agents`, 13-agent roster)

Both restricted to the same config: `Timer=30, ActInterval=0.1, Round=BestOf3, SkillLeft=Boost, SkillRight=Boost`.

Note this is a *different* metric family than the Actual-vs-Target charts above — those read `PacingSegment` rows the runtime only logs while the filter is actively steering a match, which the unfiltered runs never produce (verified: `local_setup/simulated-13agents` parquet files only ever contain `Category in {"Action", "Collision"}`).


## Prepare Data

In [ ]:
for filtered_bot, unfiltered_bot in bot_pairs:
    batch_process_pacing(
        converted_dir,
        batch_size=10,
        checkpoint_dir=comparison_checkpoint_dir,
        time_bin_size={pacing_factor_bin_size: None},  # raw factors - both sides use the same formula, no cross-run rescale needed
        bot_option=filtered_bot,
        input_format="auto",
        config_filter=pacing_config_filter,
        name=f"filtered_{filtered_bot}",
    )
    batch_process_pacing(
        converted_dir,
        batch_size=10,
        checkpoint_dir=comparison_checkpoint_dir,
        time_bin_size={pacing_factor_bin_size: None},
        bot_option=unfiltered_bot,
        input_format="auto",
        config_filter=pacing_config_filter,
        name=f"unfiltered_{unfiltered_bot}",
    )

from compile.generator import load_filtered_target_tracking

df_target_tracking = load_filtered_target_tracking(
    filtered_pacing_segments_dir,
    bots=[b for b, _ in bot_pairs],
    config_filter=pacing_config_filter,
)


In [ ]:
frames = []
for filtered_bot, unfiltered_bot in bot_pairs:
    filtered_df = load_bot_pacing_factors(
        comparison_checkpoint_dir, f"filtered_{filtered_bot}", filtered_bot,
        time_bin_size=pacing_factor_bin_size, group_label="Filtered", canonical_bot=filtered_bot,
    )
    if filtered_df is not None:
        frames.append(filtered_df)

    unfiltered_df = load_bot_pacing_factors(
        comparison_checkpoint_dir, f"unfiltered_{unfiltered_bot}", unfiltered_bot,
        time_bin_size=pacing_factor_bin_size, group_label="Unfiltered", canonical_bot=filtered_bot,
    )
    if unfiltered_df is not None:
        frames.append(unfiltered_df)

df_pacing_factor_comparison = pl.concat(frames, how="diagonal_relaxed") if frames else None
df_pacing_factor_comparison


## Filtered Bots — Data Preview

Quick, detailed look at the "Filtered" (dynamic pacing filter applied) data everything below is built from, before diving into the charts - every available column shown, nothing truncated.


### Raw Pacing Factors (`df_pacing_factor_comparison`, Group == "Filtered")

Per-(Bot, GameIndex, RoundIndex, TimeBin) row of the 8 offline-recomputed Threat/Tempo factors, restricted to the bots that had the filter applied.


In [ ]:
# Show every column and more rows instead of polars' default truncation - persists for the rest of the session.
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(80)

df_pacing_factor_filtered = (
    df_pacing_factor_comparison
    .filter(pl.col("Group") == "Filtered")
    .sort(["Bot", "GameIndex", "RoundIndex", "TimeBin"])
)
# Bot/Group sit at the end of the source schema - move them to the front since
# they're the row-identifying labels, not just more data columns.
label_cols = [c for c in ("Bot", "Group") if c in df_pacing_factor_filtered.columns]
df_pacing_factor_filtered = df_pacing_factor_filtered.select(
    label_cols + [c for c in df_pacing_factor_filtered.columns if c not in label_cols]
)
print(f"{df_pacing_factor_filtered.height:,} rows x {df_pacing_factor_filtered.width} cols")

pacing_factor_filtered_csv = os.path.join(output_dir, "pacing_factor_filtered.csv")
df_pacing_factor_filtered.write_csv(pacing_factor_filtered_csv)
print(f"✅ Saved: {pacing_factor_filtered_csv}")

df_pacing_factor_filtered

### Pacing Data

Per-segment `PacingSegment` rows - only ever logged while the filter is actively steering, so this is inherently "Filtered"-only data. Includes `PacingTarget`/`PacingConstraint` (which curve/constraint each row was steering towards), `ConfigFolder`/`GameIndex`/`Won` (real-game identity/outcome), and the engine's own Actual/Target Threat/Tempo/OverallPacing scaled values.


In [ ]:
import pandas as pd
from IPython.display import display
from plotting.pacing_filter_comparison import _to_pandas, _get_timer, _linear_trend, METRIC_TO_SCALED

df_target_tracking_detail = df_target_tracking.sort(
    ["Bot", "PacingTarget", "PacingConstraint", "ConfigFolder", "GameIndex", "TimeBin"]
).to_pandas()

# Per (Bot, PacingTarget, PacingConstraint) MAE (+ std of the pointwise
# |Actual-Target| error) for each metric, broadcast onto every row of that group -
# so the raw per-segment table also carries each group's own tracking-quality
# summary, formatted "{mae} ({std})" like the segment tables below. dropna=False
# keeps rows with a null PacingConstraint (older batches) in their own group
# instead of silently dropping them.
for metric in ["Threat", "Tempo", "OverallPacing"]:
    actual_col, target_col = METRIC_TO_SCALED[metric]
    abs_error = (df_target_tracking_detail[actual_col] - df_target_tracking_detail[target_col]).abs()
    grouped = abs_error.groupby(
        [df_target_tracking_detail[c] for c in ("Bot", "PacingTarget", "PacingConstraint")], dropna=False
    )
    mean_str = grouped.transform("mean").map("{:.3f}".format)
    std_str = grouped.transform("std").fillna(0.0).map("{:.3f}".format)
    df_target_tracking_detail[f"{metric}MAE"] = mean_str + " (" + std_str + ")"

# Drop the "Scaled" suffix for display/export - these are already the min-max
# normalized [0, 1] Actual/Target values (see compile.generator.
# batch_process_pacing_segments), the only version of these columns present in
# df_target_tracking, so the suffix is redundant noise here rather than
# disambiguating anything.
df_target_tracking_detail = df_target_tracking_detail.rename(columns={
    "ActualTempoScaled": "ActualTempo",
    "ActualThreatScaled": "ActualThreat",
    "ActualOverallPacingScaled": "ActualOverallPacing",
    "TargetTempoScaled": "TargetTempo",
    "TargetThreatScaled": "TargetThreat",
    "TargetOverallPacingScaled": "TargetOverallPacing",
})

print(f"{len(df_target_tracking_detail):,} rows x {df_target_tracking_detail.shape[1]} cols")

target_tracking_detail_csv = os.path.join(output_dir, "target_tracking_detail.csv")
df_target_tracking_detail.to_csv(target_tracking_detail_csv, index=False)
print(f"✅ Saved: {target_tracking_detail_csv}")

display(df_target_tracking_detail)

### Bot x Segment data

In [ ]:
def build_segment_mae_table(df_target_tracking, metric):
    """
    Bot x Segment table of `metric` tracking error - columns are the 1s segments
    1..Timer (see compile.generator.load_filtered_target_tracking's TimeBin), rows
    are applied bots. Each segment cell pools every PacingTarget/PacingConstraint/
    game for that (Bot, TimeBin) into "{mean |Actual-Target|} ({std})" - same
    pointwise pooling convention as _draw_target_tracking_panels. The rightmost
    "Avg" column instead holds "{overall MAE} ({trend})", where trend is the slope
    (error change per segment) of a linear fit through the per-segment means (see
    _linear_trend) - positive means tracking error grows over the match, negative
    means it shrinks.
    """
    df = _to_pandas(df_target_tracking)
    actual_col, target_col = METRIC_TO_SCALED[metric]
    timer = int(_get_timer(df))
    segments = list(range(1, timer + 1))

    df = df.assign(AbsError=(df[actual_col] - df[target_col]).abs())

    rows = {}
    for bot in sorted(df["Bot"].dropna().unique()):
        bot_df = df[df["Bot"] == bot]
        per_segment_means = []
        cells = {}
        for s in segments:
            seg_errors = bot_df.loc[bot_df["TimeBin"] == s, "AbsError"].dropna()
            if seg_errors.empty:
                cells[str(s)] = ""
                continue
            mean = seg_errors.mean()
            std = seg_errors.std(ddof=1) if len(seg_errors) > 1 else 0.0
            cells[str(s)] = f"{mean:.3f} ({std:.3f})"
            per_segment_means.append((s, mean))

        overall_mae = bot_df["AbsError"].dropna().mean()
        trend = _linear_trend([s for s, _ in per_segment_means], [m for _, m in per_segment_means])
        slope = trend[0] if trend is not None else float("nan")
        cells["Avg"] = f"{overall_mae:.3f} ({slope:.3f})"
        rows[bot] = cells

    table = pd.DataFrame.from_dict(rows, orient="index")[[str(s) for s in segments] + ["Avg"]]
    table.index.name = "Bot"
    return table


for metric in ["Threat", "Tempo", "OverallPacing"]:
    segment_mae_table = build_segment_mae_table(df_target_tracking, metric)
    segment_mae_csv = os.path.join(output_dir, f"segment_mae_{metric.lower()}.csv")
    segment_mae_table.to_csv(segment_mae_csv)
    print(f"✅ Saved: {segment_mae_csv}")
    display(segment_mae_table)

## Pacing Factor Breakdown

In [ ]:
from plotting.pacing_filter_comparison import plot_all_pacing_factor_comparisons

figs_factor_comparison = plot_all_pacing_factor_comparisons(
    df_pacing_factor_comparison, output_dir=comparison_output_dir,
)


## Plot

### Pacing Over Time (per 1s Segment)

Same data as above, but broken out by `TimeBin` instead of pooled — one point per 1-second segment across the match (30 segments for `Timer=30`), so divergence *when* in the match it happens is visible, not just the aggregate shift.


In [ ]:
from plotting.pacing_filter_comparison import plot_all_pacing_factor_timeseries

figs_factor_timeseries = plot_all_pacing_factor_timeseries(
    df_pacing_factor_comparison, output_dir=comparison_output_dir,
)


### Overall Pacing (Merged Threat/Tempo Composite)

Merges the 8 raw factors above into a single **Threat**, **Tempo**, and **OverallPacing** score per row — min-max normalized so all 8 factors are on the same [0, 1] scale, then averaged per the engine's own Threat/Tempo grouping (Threat = CollisionRatio/AbilityRatio/Angle/SafeDistance, Tempo = ActionIntensity/ActionDensity/BotsDistance/Velocity, Overall = mean of the two).

**Caveat**: this is an equal-weight approximation, not the engine's real internal formula — those weights aren't available offline, only the 8 raw factors themselves. Useful for a condensed at-a-glance comparison, but don't read exact values as the "true" pacing score the way `ActualOverallPacingScaled` is for the filtered-only Actual-vs-Target charts above.


In [ ]:
from plotting.pacing_filter_comparison import plot_all_overall_pacing_timeseries

figs_overall_pacing = plot_all_overall_pacing_timeseries(
    df_pacing_factor_comparison, output_dir=comparison_output_dir,
)


### Filtered Actual vs Target (Engine Ground Truth)

`pacing_fix3`'s `PacingSegment` rows already carry the engine's own computed `Tempo`/`Threat`/`OverallPacing` and `Target*` curves — this swaps the Filtered side from the equal-weight offline approximation above to that ground truth, and overlays the predefined Target curve it was steering towards.

One figure per (Bot, PacingTarget) per constraint — there are 5 target curve shapes (`default_target_low/med/high`, `linear_increase`, `linear_decrease`); averaging them together would cancel out the shape, so each gets its own chart. `LocalSegmentIndex` isn't a fixed 1s bin (segment count varies with how long a round ran), so it's converted to elapsed seconds via `RoundDuration / NumSegmentsInRound` to land on the same "Segment (s)" axis as the charts above.

Each panel shows: **Target** (predefined curve), **Filtered Actual** (mean ± std band across matching rounds, plus a linear trend line), **MSE(Actual, Target)** annotated, and the **Unfiltered** offline-approximated line from before as light context (different scale/formula — not a strict apples-to-apples number, directional only).

Split into one sub-section per `PacingConstraint` (`avg_bot`, `top_5`, `nn` on prod - see `compile.log_to_parquet.parse_pacing_folder_name`), same as the Merged section below, so each constraint's per-bot tracking charts land in their own cell/output instead of one dump mixing all of them together.


#### avg_bot

In [ ]:
from plotting.pacing_filter_comparison import plot_all_filtered_target_tracking

pacing_constraint = "avg_bot"
df_target_tracking_avg_bot = df_target_tracking.filter(pl.col("PacingConstraint") == pacing_constraint)

figs_target_tracking_avg_bot = plot_all_filtered_target_tracking(
    df_pacing_factor_comparison, df_target_tracking_avg_bot,
    output_dir=os.path.join(comparison_output_dir, pacing_constraint),
    sim_targets_dir=sim_targets_dir,
)

#### top_5

In [ ]:
pacing_constraint = "top_5"
df_target_tracking_top_5 = df_target_tracking.filter(pl.col("PacingConstraint") == pacing_constraint)

figs_target_tracking_top_5 = plot_all_filtered_target_tracking(
    df_pacing_factor_comparison, df_target_tracking_top_5,
    output_dir=os.path.join(comparison_output_dir, pacing_constraint),
    sim_targets_dir=sim_targets_dir,
)

#### nn

In [ ]:
pacing_constraint = "nn"
df_target_tracking_nn = df_target_tracking.filter(pl.col("PacingConstraint") == pacing_constraint)

figs_target_tracking_nn = plot_all_filtered_target_tracking(
    df_pacing_factor_comparison, df_target_tracking_nn,
    output_dir=os.path.join(comparison_output_dir, pacing_constraint),
    sim_targets_dir=sim_targets_dir,
)

### Merged (Both Applied Bots Pooled)

Same Actual-vs-Target tracking, but MCTS and NN pooled into one line per `PacingTarget` instead of a separate chart each — 5 figures total instead of 10 per constraint. Mirrors how `plotting.pacing_target_analyzer.plot_pacing_target_tracking` already pools every "Applied"-role bot together by default.

Split into one sub-section per `PacingConstraint` (parsed from the config folder name — `Pacing_<target>_constraint_<constraint>` — alongside `PacingTarget`, see `compile.log_to_parquet.parse_pacing_folder_name`; prod batches carry `avg_bot`, `top_5`, `nn`) so each constraint's pooled tracking charts land in their own cell/output instead of one dump mixing all of them together.


#### avg_bot

In [ ]:
from plotting.pacing_filter_comparison import plot_all_filtered_target_tracking_merged

pacing_constraint = "avg_bot"
df_target_tracking_avg_bot = df_target_tracking.filter(pl.col("PacingConstraint") == pacing_constraint)

figs_target_tracking_merged_avg_bot = plot_all_filtered_target_tracking_merged(
    df_pacing_factor_comparison, df_target_tracking_avg_bot,
    output_dir=os.path.join(comparison_output_dir, pacing_constraint),
    sim_targets_dir=sim_targets_dir,
)

#### top_5

In [ ]:
pacing_constraint = "top_5"
df_target_tracking_top_5 = df_target_tracking.filter(pl.col("PacingConstraint") == pacing_constraint)

figs_target_tracking_merged_top_5 = plot_all_filtered_target_tracking_merged(
    df_pacing_factor_comparison, df_target_tracking_top_5,
    output_dir=os.path.join(comparison_output_dir, pacing_constraint),
    sim_targets_dir=sim_targets_dir,
)

#### nn

In [ ]:
pacing_constraint = "nn"
df_target_tracking_nn = df_target_tracking.filter(pl.col("PacingConstraint") == pacing_constraint)

figs_target_tracking_merged_nn = plot_all_filtered_target_tracking_merged(
    df_pacing_factor_comparison, df_target_tracking_nn,
    output_dir=os.path.join(comparison_output_dir, pacing_constraint),
    sim_targets_dir=sim_targets_dir,
)

### Per-Bot (Not Pooled)

Same Actual-vs-Target tracking, but each applied bot (MCTS/NN) gets its own Filtered/Unfiltered/Trend lines instead of being averaged into one line - does each bot track the curve differently, rather than one pooled answer. `include_target`/`include_unfiltered`/`include_trend` toggle each line type off across every figure.

Split into one sub-section per `PacingConstraint` (`avg_bot`, `top_5`, `nn` on prod - see `compile.log_to_parquet.parse_pacing_folder_name`), same as the sections above, so each constraint's per-bot tracking charts land in their own cell/output instead of one dump mixing all of them together.


#### avg_bot

In [ ]:
from plotting.pacing_filter_comparison import plot_all_filtered_target_tracking_by_bot

pacing_constraint = "avg_bot"
df_target_tracking_avg_bot = df_target_tracking.filter(pl.col("PacingConstraint") == pacing_constraint)

figs_target_tracking_by_bot_avg_bot = plot_all_filtered_target_tracking_by_bot(
    df_pacing_factor_comparison, df_target_tracking_avg_bot,
    output_dir=os.path.join(comparison_output_dir, pacing_constraint),
    sim_targets_dir=sim_targets_dir,
    include_target=True, include_unfiltered=True, include_trend=True,
)

#### top_5

In [ ]:
pacing_constraint = "top_5"
df_target_tracking_top_5 = df_target_tracking.filter(pl.col("PacingConstraint") == pacing_constraint)

figs_target_tracking_by_bot_top_5 = plot_all_filtered_target_tracking_by_bot(
    df_pacing_factor_comparison, df_target_tracking_top_5,
    output_dir=os.path.join(comparison_output_dir, pacing_constraint),
    sim_targets_dir=sim_targets_dir,
    include_target=True, include_unfiltered=True, include_trend=True,
)

#### nn

In [ ]:
pacing_constraint = "nn"
df_target_tracking_nn = df_target_tracking.filter(pl.col("PacingConstraint") == pacing_constraint)

figs_target_tracking_by_bot_nn = plot_all_filtered_target_tracking_by_bot(
    df_pacing_factor_comparison, df_target_tracking_nn,
    output_dir=os.path.join(comparison_output_dir, pacing_constraint),
    sim_targets_dir=sim_targets_dir,
    include_target=True, include_unfiltered=True, include_trend=True,
)

### Tracking Error vs Target Curve Shape (Bot x Curve Archetype)

Instead of treating each `PacingTarget` config as an identity to name/render (doesn't scale past a handful of configs), both charts below derive shape features straight from the deterministic curve itself (`compute_target_curve_features`: Volatility, Amplitude, Trend, DirectionChanges) and plot/bucket by those instead:

- **Scatter**: x = target curve Volatility, y = tracking MAE, one point per (Bot, PacingTarget), colored by Bot, with a linear trend line - does tracking error actually scale with how erratic the target curve is? No PacingTarget name is ever drawn, so this scales to any number of configs. Both axes use fixed-width 0.1 bins; `bin_width`/`y_bin_width` set how many of those bins to show (x-axis spans `[0, bin_width*0.1]`, y-axis spans `[0, y_bin_width*0.1]` - default 10 each, i.e. the full `[0, 1]`), consistent across all 3 panels.
- **Heatmap**: each `PacingTarget` is auto-bucketed into a curve Archetype (`Flat` / `Rising` / `Falling`, from the curve's own amplitude/trend stats - a curve nets into Rising/Falling by its overall trend direction regardless of how much it oscillates along the way), then Bot x Archetype mean MAE is shown - a small, fixed-size grid regardless of how many raw target files exist. All 3 archetype columns are always shown, even ones with no matching curves in a given run/metric, and the color scale uses the same fixed-width 0.1 `bin_width`-count convention (default 10, i.e. `[0, 1]`) instead of auto-scaling per panel.


In [ ]:
from plotting.pacing_filter_comparison import (
    plot_target_tracking_error_vs_volatility,
    plot_target_tracking_error_by_archetype_heatmap,
)

fig_error_vs_volatility = plot_target_tracking_error_vs_volatility(df_target_tracking, sim_targets_dir, bin_width=5, y_bin_width=10)
fig_error_by_archetype = plot_target_tracking_error_by_archetype_heatmap(df_target_tracking, sim_targets_dir, bin_width=5)


### Tracking Error vs Win Rate

Does tracking the target curve more closely (lower MAE) actually correlate with winning more? Bins every applied bot's individual game by its MAE (see `compute_game_level_tracking_error` - one row per real game, keyed by `(Bot, PacingTarget, ConfigFolder, GameIndex)` since `GameIndex` alone only numbers games *within* one `ConfigFolder`) into `bin_width` fixed-width 0.1 bins (x-axis spans `[0, bin_width*0.1]` - default 10, i.e. the full normalized `[0, 1]` MAE range) and plots each bin's win rate, with the point-biserial correlation `r` annotated per panel.

> Needs `df_target_tracking` to include `ConfigFolder`/`GameIndex`/`Won` - re-run the "Prepare Data" cell above (`load_filtered_target_tracking`) if it was loaded before this was added.


In [ ]:
from plotting.pacing_filter_comparison import plot_tracking_error_vs_winrate

fig_error_vs_winrate = plot_tracking_error_vs_winrate(df_target_tracking, bin_width=10)
